# NB 05: Backtesting

Event-driven backtest of cross-exchange funding rate arbitrage.
Grid search over entry thresholds, leverage, and exchange pairs.
70/30 in-sample / out-of-sample chronological split.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

from src.config import EXCHANGES, ASSETS, PROCESSED_DIR, FIGURES_DIR
from src.models.backtester import (
    BacktestConfig, run_backtest, run_backtest_grid, results_to_dataframe,
)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sns.set_style('whitegrid')

# Load aligned wide-format funding rates
wide = pd.read_parquet(PROCESSED_DIR / 'funding_rates_aligned.parquet')
print(f'Data shape: {wide.shape}')
print(f'Date range: {wide.index.min()} to {wide.index.max()}')

# Available exchange-asset pairs
available = [(ex, a) for ex, a in wide.columns]
print(f'Available series: {len(available)}')

## 1. Grid Search: All Viable Pairs

In [ ]:
all_results = []

entry_thresholds = [0.0001, 0.00025, 0.0005, 0.001]
leverage_levels = [1, 2, 3, 5]

for asset in sorted(wide.columns.get_level_values('asset').unique()):
    exchanges_for_asset = sorted(set(e for e, a in wide.columns if a == asset))
    
    for ex_a, ex_b in combinations(exchanges_for_asset, 2):
        print(f'Running backtest: {ex_a}/{ex_b} {asset}...', end=' ')
        
        try:
            results = run_backtest_grid(
                wide, ex_a, ex_b, asset,
                entry_thresholds=entry_thresholds,
                leverage_levels=leverage_levels,
            )
            all_results.extend(results)
            n_trades = sum(r.n_trades for r in results)
            print(f'{len(results)} configs, {n_trades} total trades')
        except Exception as e:
            print(f'ERROR: {e}')

print(f'\nTotal backtest runs: {len(all_results)}')

## 2. Results Summary

In [ ]:
results_df = results_to_dataframe(all_results)

# In-sample performance
is_results = results_df[results_df['sample_type'] == 'in_sample'].copy()
os_results = results_df[results_df['sample_type'] == 'out_of_sample'].copy()

print(f'In-sample results: {len(is_results)} configs')
print(f'Out-of-sample results: {len(os_results)} configs')

# Top 10 in-sample by Sharpe
if not is_results.empty:
    top_is = is_results.nlargest(10, 'sharpe_ratio')
    display_cols = ['exchange_a', 'exchange_b', 'asset', 'n_trades',
                    'total_return', 'annualized_return', 'sharpe_ratio',
                    'max_drawdown', 'config_entry_threshold', 'config_leverage']
    display_cols = [c for c in display_cols if c in top_is.columns]
    print('\nTop 10 In-Sample by Sharpe Ratio:')
    top_is[display_cols].round(4)

## 3. Out-of-Sample Validation

In [ ]:
# Compare IS vs OOS performance for top IS configs
if not is_results.empty and not os_results.empty:
    # Merge IS and OOS on config keys
    merge_keys = ['exchange_a', 'exchange_b', 'asset',
                  'config_entry_threshold', 'config_leverage']
    merge_keys = [k for k in merge_keys if k in is_results.columns and k in os_results.columns]
    
    if merge_keys:
        comparison = is_results.merge(
            os_results, on=merge_keys, suffixes=('_is', '_os')
        )
        
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        # Sharpe ratio comparison
        if 'sharpe_ratio_is' in comparison.columns:
            axes[0].scatter(comparison['sharpe_ratio_is'], comparison['sharpe_ratio_os'], alpha=0.5)
            axes[0].plot([comparison['sharpe_ratio_is'].min(), comparison['sharpe_ratio_is'].max()],
                        [comparison['sharpe_ratio_is'].min(), comparison['sharpe_ratio_is'].max()],
                        'r--', alpha=0.5)
            axes[0].set_xlabel('In-Sample Sharpe')
            axes[0].set_ylabel('Out-of-Sample Sharpe')
            axes[0].set_title('Sharpe Ratio: IS vs OOS')
        
        # Total return comparison
        if 'total_return_is' in comparison.columns:
            axes[1].scatter(comparison['total_return_is'], comparison['total_return_os'], alpha=0.5)
            axes[1].plot([0, comparison['total_return_is'].max()],
                        [0, comparison['total_return_is'].max()], 'r--', alpha=0.5)
            axes[1].set_xlabel('In-Sample Return')
            axes[1].set_ylabel('Out-of-Sample Return')
            axes[1].set_title('Total Return: IS vs OOS')
        
        # Max drawdown comparison
        if 'max_drawdown_is' in comparison.columns:
            axes[2].scatter(comparison['max_drawdown_is'], comparison['max_drawdown_os'], alpha=0.5)
            axes[2].set_xlabel('In-Sample Max DD')
            axes[2].set_ylabel('Out-of-Sample Max DD')
            axes[2].set_title('Max Drawdown: IS vs OOS')
        
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / 'is_vs_oos.pdf', bbox_inches='tight')
        plt.show()

## 4. Equity Curves — Best Configurations

In [ ]:
# Plot equity curves for top 5 configs (full period)
# Re-run top configs on full data
if not is_results.empty:
    top5 = is_results.nlargest(5, 'sharpe_ratio')
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    for _, row in top5.iterrows():
        config = BacktestConfig(
            entry_threshold=row.get('config_entry_threshold', 0.0005),
            leverage=row.get('config_leverage', 1),
        )
        result = run_backtest(
            wide, row['exchange_a'], row['exchange_b'], row['asset'],
            config=config, sample_type='full',
        )
        
        if result.equity_curve is not None:
            label = f"{row['exchange_a']}/{row['exchange_b']} {row['asset']} (SR={row['sharpe_ratio']:.2f})"
            ax.plot(result.equity_curve.index, result.equity_curve.values, label=label)
    
    ax.set_title('Equity Curves — Top 5 Configurations')
    ax.set_ylabel('Cumulative Return (1 = start)')
    ax.legend()
    ax.axhline(y=1, color='k', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'equity_curves_top5.pdf', bbox_inches='tight')
    plt.show()

## 5. Parameter Sensitivity Heatmap

In [ ]:
# Heatmap: Sharpe by entry threshold x leverage (best pair)
if not os_results.empty:
    # Use OOS results for less biased view
    best_pair = os_results.loc[os_results['sharpe_ratio'].idxmax()]
    pair_mask = (
        (os_results['exchange_a'] == best_pair['exchange_a']) &
        (os_results['exchange_b'] == best_pair['exchange_b']) &
        (os_results['asset'] == best_pair['asset'])
    )
    pair_results = os_results[pair_mask]
    
    if 'config_entry_threshold' in pair_results.columns and 'config_leverage' in pair_results.columns:
        pivot = pair_results.pivot_table(
            index='config_entry_threshold',
            columns='config_leverage',
            values='sharpe_ratio',
        )
        
        fig, ax = plt.subplots(figsize=(8, 5))
        sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', center=0, ax=ax)
        ax.set_title(f'OOS Sharpe: {best_pair["exchange_a"]}/{best_pair["exchange_b"]} {best_pair["asset"]}')
        ax.set_ylabel('Entry Threshold')
        ax.set_xlabel('Leverage')
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / 'parameter_sensitivity.pdf', bbox_inches='tight')
        plt.show()

In [ ]:
# Save results
if not results_df.empty:
    results_df.to_parquet(PROCESSED_DIR / 'backtest_results.parquet', index=False)
    print(f'Saved {len(results_df)} backtest results to {PROCESSED_DIR / "backtest_results.parquet"}')